### Assignment 2


#### Assignment 2.1 — Shape Prediction and Axis Semantics
- **Objective**: Accurately predict and categorize tensor shape transformations without running code.
- **Given**: Base tensor `x = torch.zeros(2, 3, 4, 5)`.
- **Build / Do**:
  1. Write on paper the predicted output shapes and categorize each transformation (removed, inserted, reordered, merged, or enlarged):
     - `x[0]`
     - `x[0:1]`
     - `x[..., -1]`
     - `x[:, None, :, :, :]`
     - `x.permute(0, 2, 3, 1)`
     - `x.flatten(1, 2)`
     - `torch.stack([x, x], dim=2)`
     - `torch.cat([x, x], dim=1)`
  2. Implement assertion checks for all 8 operations.
- **Pass Criteria**:
  - Assert exact shapes:
    - `x[0].shape == (3, 4, 5)` (axis 0 removed)
    - `x[0:1].shape == (1, 3, 4, 5)` (axis 0 preserved as slice)
    - `x[..., -1].shape == (2, 3, 4)` (last axis removed)
    - `x[:, None, :, :, :].shape == (2, 1, 3, 4, 5)` (axis 1 inserted)
    - `x.permute(0, 2, 3, 1).shape == (2, 4, 5, 3)` (axes reordered)
    - `x.flatten(1, 2).shape == (2, 12, 5)` (axes 1 and 2 merged)
    - `torch.stack([x, x], dim=2).shape == (2, 3, 2, 4, 5)` (new axis inserted at dim 2)
    - `torch.cat([x, x], dim=1).shape == (2, 6, 4, 5)` (axis 1 enlarged from 3 to 6)


#### Assignment 2.2 — Broadcasting Debugger & Loss Bug Analysis
- **Objective**: Master right-to-left broadcasting rules and eliminate silent loss calculation bugs.
- **Given**: Four shape pairs:
  1. `[4, 1, 8]` and `[3, 8]`
  2. `[2, 3]` and `[3, 2]`
  3. `[5, 1]` and `[7]`
  4. `[2, 3, 4]` and `[3, 1]`
- **Build / Do**:
  1. Right-align each pair, check compatibility for each axis, and predict the output shape or `RuntimeError`.
  2. Verify with code:
     - Pair 1: `(4, 3, 8)`
     - Pair 2: Incompatible (raises `RuntimeError: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1`)
     - Pair 3: `(5, 7)`
     - Pair 4: `(2, 3, 4)`
  3. Recreate the common loss bug: predictions `pred = torch.randn(32, 1)` and targets `target = torch.randn(32)`. Show that `(pred - target).shape == (32, 32)`, and write the fix using `unsqueeze(1)`.
- **Pass Criteria**:
  - All 4 predictions validated by executable code.
  - Written explanation of why `(pred - target)` produces a $32 \times 32$ matrix instead of a $32 \times 1$ vector and how this corrupts MSE or L1 loss.


#### Assignment 2.3 — Indexing Laboratory: Views vs. Copies
- **Objective**: Distinguish basic indexing (views) from advanced indexing (copies) and master multidimensional selections.
- **Given**: `x = torch.arange(20).reshape(4, 5)`.
- **Build / Do**:
  1. Select rows `3, 1, 1` using integer tensor indexing: `x[torch.tensor([3, 1, 1])]`.
  2. Select columns `4, 0, 2`: `x[:, torch.tensor([4, 0, 2])]`.
  3. Select coordinate pairs $(0, 4)$, $(2, 0)$, and $(3, 2)$: `x[torch.tensor([0, 2, 3]), torch.tensor([4, 0, 2])]`.
  4. Select the Cartesian submatrix of rows `[0, 3]` and columns `[1, 2, 4]` using broadcasted index vectors.
  5. Select all elements divisible by 3 using a boolean mask: `x[x % 3 == 0]`.
  6. Prove that basic slice `x[0, :]` shares memory (mutating it changes `x`), while advanced indexing `x[torch.tensor([0])]` produces a copy.
- **Pass Criteria**:
  - Assert exact shapes and values for all 5 selections:
    - Rows `[3, 1, 1]`: shape `(3, 5)`
    - Columns `[4, 0, 2]`: shape `(4, 3)`
    - Coordinate pairs: shape `(3,)`, values `[4, 10, 17]`
    - Cartesian block: shape `(2, 3)`, values `[[1, 2, 4], [16, 17, 19]]`
    - Divisible by 3: shape `(7,)`, values `[0, 3, 6, 9, 12, 15, 18]`
  - Explicit assertion confirming mutation of basic slice affects `x` and mutation of advanced index does not.


#### Assignment 2.4 — Gather, Scatter, and Grouped Reductions
- **Objective**: Implement indexed extraction, one-hot encoding, and histogram counting using `gather` and `scatter`.
- **Given**:
  - Logits matrix: `logits = torch.randn(6, 4)`
  - Class targets: `targets = torch.tensor([1, 0, 3, 2, 1, 0], dtype=torch.long)`
- **Build / Do**:
  1. Use `gather(dim=1, index=targets[:, None])` to extract the predicted logit corresponding to the true target for each sample.
  2. Use `torch.zeros(6, 4).scatter_(dim=1, index=targets[:, None], src=1.0)` to generate a one-hot label matrix.
  3. Use `torch.zeros(4).scatter_add_(dim=0, index=targets, src=torch.ones(6))` to count the frequency of each class.
- **Pass Criteria**:
  - Assert that `gathered[:, 0]` matches manual loop extraction `torch.tensor([logits[i, targets[i]] for i in range(6)])`.
  - Assert that one-hot matrix matches `F.one_hot(targets, num_classes=4).float()`.
  - Assert that class counts match `torch.bincount(targets, minlength=4).float()`.


#### Assignment 2.5 — Vectorized Sliding-Window Moving Average
- **Objective**: Compute moving averages efficiently using `unfold` without Python loops.
- **Given**: 1D sequence `x = torch.arange(1, 11, dtype=torch.float32)` (values 1 through 10), window size $W = 3$.
- **Build / Do**:
  1. Use `x.unfold(dimension=0, size=3, step=1)` to generate consecutive sliding windows.
  2. Compute the mean along the unfolded window dimension (`dim=1`).
- **Pass Criteria**:
  - No Python loops or list comprehensions used.
  - Assert output shape is `(8,)`.
  - Assert exact values: `torch.testing.assert_close(result, torch.tensor([2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]))`.


#### Assignment 2.6 — Einstein Summation vs. Standard Operations
- **Objective**: Translate tensor contractions between mathematical notation, `torch.einsum`, and standard batched matrix multiplication (`@`).
- **Given**:
  - Feature tensor `x` of shape `[B, T, D] = [2, 5, 8]`
  - Linear weight `W` of shape `[D, O] = [8, 16]` and bias `b` of shape `[O] = [16]`
  - Query/Key tensors `q, k` of shape `[B, H, T, Dh] = [2, 4, 5, 16]`
- **Build / Do**:
  1. Compute linear projection with `einsum`: `torch.einsum("btd,do->bto", x, W) + b`.
  2. Compute linear projection with `@`: `(x @ W) + b`.
  3. Compute unscaled attention scores with `einsum`: `torch.einsum("bhtd,bhsd->bhts", q, k)`.
  4. Compute attention scores with `@`: `q @ k.transpose(-2, -1)`.
- **Pass Criteria**:
  - Assert `torch.allclose` between `einsum` and `@` linear outputs.
  - Assert `torch.allclose` between `einsum` and `@` attention scores of shape `[2, 4, 5, 5]`.
  - Provide written explanation identifying which indices are contracted (reduced) in each equation (`d` in linear, `Dh` in attention).


#### Assignment 2.7 — Storage, Strides, and Contiguity Investigation
- **Objective**: Inspect the physical memory representation of tensor views, transpositions, and copies.
- **Given**: `x = torch.arange(24).reshape(4, 6)`.
- **Build / Do**:
  1. Create:
     - Slice: `s = x[:2, :3]`
     - Transpose: `t = x.t()`
     - Advanced index: `a = x[[0, 1], :]`
     - Cloned copy: `c = x.clone()`
     - Detached view: `d = x.detach()`
     - Expanded view: `e = x[0:1, :].expand(5, 6)`
  2. For each tensor, record: `shape`, `stride()`, `storage_offset()`, `is_contiguous()`, and whether in-place mutation alters `x`.
- **Pass Criteria**:
  - Produce a structured table recording these 5 properties for all cases.
  - Verify that `t.stride() == (1, 6)` and `t.is_contiguous() == False`.
  - Verify that `e.stride() == (0, 1)` (zero stride on expanded dimension).
  - Verify that mutating `s` or `d` alters `x`, while mutating `a` or `c` does not.


#### Assignment 2.8 — Patch Extraction: Native Operations vs. Einops
- **Objective**: Implement Vision Transformer (ViT) patch extraction and spatial reconstruction using native PyTorch operations.
- **Given**: Image batch `images = torch.randn(4, 3, 32, 32)` and patch size $P = 8$ (grid size $H/P = 4, W/P = 4$, total patches = 16, patch dimension = $3 \times 8 \times 8 = 192$).
- **Build / Do**:
  1. Reshape and permute `images` to patch representation of shape `[4, 16, 192]` using only native `reshape` and `permute`.
  2. Invert the transformation to reconstruct the original `[4, 3, 32, 32]` tensor from patches.
  3. (Optional/Reference): Write the equivalent `einops` pattern string (`"b c (h p1) (w p2) -> b (h w) (c p1 p2)"`).
- **Pass Criteria**:
  - Assert patch tensor shape is exactly `(4, 16, 192)`.
  - Assert reconstructed tensor matches input: `torch.testing.assert_close(reconstructed, images)`.


#### Assignment 2.9 — Per-Channel Image Normalization Mechanics
- **Objective**: Implement per-channel batch normalization from scratch using multi-axis reductions, dimension preservation, and broadcasting.
- **Given**:
  - Random image batch: `images = torch.rand(16, 3, 64, 64, dtype=torch.float32)`
  - Degenerate constant batch: `constant_images = torch.full((4, 3, 16, 16), 5.0, dtype=torch.float32)`
- **Build / Do**:
  1. Compute per-channel mean and standard deviation (`correction=0`) across batch, height, and width (`dim=(0, 2, 3)`) preserving dimensions with `keepdim=True`.
  2. Guard against division by zero using `clamp_min(1e-6)`.
  3. Normalize the batch via broadcasted subtraction and division.
  4. Repeat normalization on `constant_images` and demonstrate that clamping prevents `NaN` and `Inf`.
  5. Scientifically verify that the normalized output for `images` has per-channel mean $\approx 0.0$ and std $\approx 1.0$ using `torch.testing.assert_close`.
- **Pass Criteria**:
  - Assert output shape is strictly `(16, 3, 64, 64)`.
  - Assert `torch.testing.assert_close(norm_mean, torch.zeros(3), atol=1e-5, rtol=1e-5)`.
  - Assert `torch.testing.assert_close(norm_std, torch.ones(3), atol=1e-5, rtol=1e-5)`.
  - Assert `torch.isfinite(norm_constant).all()` evaluates to `True`.

**Exit criterion:** You can predict shapes, choose basic versus advanced indexing, use gather/scatter, express contractions, and identify when an operation is a view or copy.